# ARM97 Model Output Variables With Observation Comparison

This notebook extends `ARM97_model_output_all_variables.ipynb` with observation overlays and model-minus-observation heatmaps where matching ARM97 IOP observation variables are available.

Sections:

1. Surface time series: all model surface variables are available; observation is overlaid when mapped.
2. Profile time series: all model profile variables are available; observation is overlaid at the selected pressure level when mapped.
3. Model heatmap: same model-only heatmap behavior as the reference notebook.
4. Heatmap diff: model minus observation heatmap for mapped profile variables.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime, timedelta
import os
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "configs").exists() and (candidate / "notebooks").exists():
            return candidate
    return Path("/Users/yunlong/Workshop/SCM-UQ-Workflow")


ROOT = Path(os.environ.get("SCM_UQ_WORKFLOW_ROOT", find_repo_root())).resolve()
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".local_cache/matplotlib-cache"))
os.environ.setdefault("XDG_CACHE_HOME", str(ROOT / ".local_cache"))

import numpy as np
import pandas as pd
from netCDF4 import Dataset, num2date
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output

# Manually choose files here. The widget below can also reload different paths.
CASE_DIR = Path(
    os.environ.get(
        "ARM97_CASE_DIR",
        str(ROOT / "cases/run_e3sm_scm_ARM97"),
    )
).expanduser().resolve()
MODEL_FILE = Path(
    os.environ.get(
        "ARM97_MODEL_FILE",
        str(CASE_DIR / "arm97_model_ready.nc"),
    )
).expanduser().resolve()
OBSERVATION_FILE = Path(
    os.environ.get(
        "ARM97_OBSERVATION_FILE",
        str(CASE_DIR / "arm97_iop_observation_model_window_nco.nc"),
    )
).expanduser().resolve()

OUT_DIR = CASE_DIR / "notebook_outputs" / "arm97_model_output_vs_observation_all_variables"

assert ROOT.exists(), ROOT
assert MODEL_FILE.exists(), MODEL_FILE
assert OBSERVATION_FILE.exists(), OBSERVATION_FILE

print("repo root:", ROOT)
print("model file:", MODEL_FILE)
print("observation file:", OBSERVATION_FILE)
print("output dir:", OUT_DIR)


## Variable Mapping

The notebook includes the ARM97 model-to-observation mappings used in the multi-model comparison notebook. For variables not listed here, it also tries direct same-name observation matching.


In [ ]:
@dataclass(frozen=True)
class VarSpec:
    model: str
    obs: str
    units: str
    scale_obs: float = 1.0
    obs_offset: float = 0.0
    description: str = ""


@dataclass(frozen=True)
class ProfileSpec:
    model: str
    obs: str
    units: str
    scale_obs: float = 1.0
    obs_offset: float = 0.0
    description: str = ""


SURFACE_OBS_SPECS = {
    spec.model: spec
    for spec in [
        VarSpec("TREFHT", "Tsair", "K", description="2 m air temperature"),
        VarSpec("TS", "Tg", "K", description="surface/ground temperature"),
        VarSpec("TMQ", "prew", "kg/m2", scale_obs=10.0, description="precipitable water"),
        VarSpec("CLDTOT", "totcld", "1", scale_obs=0.01, description="total cloud fraction"),
        VarSpec("CLDLOW", "lowcld", "1", scale_obs=0.01, description="low cloud fraction"),
        VarSpec("CLDMED", "midcld", "1", scale_obs=0.01, description="mid-level cloud fraction"),
        VarSpec("CLDHGH", "hghcld", "1", scale_obs=0.01, description="high cloud fraction"),
        VarSpec("PS", "Ps", "Pa", description="surface pressure"),
        VarSpec("LHFLX", "lhflx", "W/m2", description="latent heat flux"),
        VarSpec("SHFLX", "shflx", "W/m2", description="sensible heat flux"),
        VarSpec("FSNS", "srfswdn-srfswup", "W/m2", description="surface net shortwave flux"),
        VarSpec("FLNS", "srflwup-srflwdn", "W/m2", description="surface net longwave flux"),
        VarSpec("FSDS", "srfswdn", "W/m2", description="surface downwelling shortwave flux"),
        VarSpec("FLDS", "srflwdn", "W/m2", description="surface downwelling longwave flux"),
        VarSpec("FLUT", "TOA_LWup", "W/m2", description="TOA upwelling longwave flux"),
        VarSpec("U10", "windsrf", "m/s", description="10 m wind speed"),
        VarSpec("PRECT", "Prec", "m/s", scale_obs=0.001, description="total precipitation rate"),
    ]
}

PROFILE_OBS_SPECS = {
    spec.model: spec
    for spec in [
        ProfileSpec("T", "T", "K", description="temperature"),
        ProfileSpec("Q", "q", "kg/kg", description="specific humidity"),
        ProfileSpec("U", "u", "m/s", description="zonal wind"),
        ProfileSpec("V", "v", "m/s", description="meridional wind"),
        ProfileSpec("OMEGA", "omega", "Pa/s", description="pressure vertical velocity"),
        ProfileSpec("RELHUM", "rh", "%", description="relative humidity"),
    ]
}


## Inspect Model Variables


In [ ]:
VERTICAL_DIMS = {"lev", "ilev"}
OBS_SKIP_VARS = {"time", "tsec", "bdate", "lat", "lon", "lev", "year", "month", "day", "hour", "minute"}


def is_numeric_variable(var):
    return np.issubdtype(np.dtype(var.dtype), np.number)


def classify_model_variable(name, var):
    dims = tuple(var.dimensions)
    if "time" not in dims or not is_numeric_variable(var):
        return None
    if name in {"time_bnds"}:
        return None
    if any(dim in dims for dim in VERTICAL_DIMS):
        return "profile"
    return "surface"


def variable_catalog(model_file):
    rows = []
    with Dataset(model_file) as ds:
        for name, var in ds.variables.items():
            category = classify_model_variable(name, var)
            if category is None:
                continue
            rows.append(
                {
                    "variable": name,
                    "category": category,
                    "dimensions": ", ".join(var.dimensions),
                    "shape": " x ".join(str(x) for x in var.shape),
                    "units": getattr(var, "units", ""),
                    "long_name": getattr(var, "long_name", ""),
                }
            )
    if not rows:
        return pd.DataFrame(columns=["variable", "category", "dimensions", "shape", "units", "long_name"])
    return pd.DataFrame(rows).sort_values(["category", "variable"]).reset_index(drop=True)


def observation_variable_names(observation_file):
    with Dataset(observation_file) as ds:
        return set(ds.variables)


def obs_names_required(expression):
    return [name.strip() for name in expression.replace("-", ",").split(",") if name.strip()]


def spec_available(spec, obs_names):
    return spec is not None and all(name in obs_names for name in obs_names_required(spec.obs))


def direct_surface_spec(variable_name, obs_names, units=""):
    if variable_name in obs_names and variable_name not in OBS_SKIP_VARS:
        return VarSpec(variable_name, variable_name, units)
    return None


def direct_profile_spec(variable_name, obs_names, units=""):
    if variable_name in obs_names and variable_name not in OBS_SKIP_VARS:
        return ProfileSpec(variable_name, variable_name, units)
    return None


def attach_observation_mapping(catalog, observation_file):
    obs_names = observation_variable_names(observation_file)
    rows = []
    for _, row in catalog.iterrows():
        name = row["variable"]
        if row["category"] == "surface":
            spec = SURFACE_OBS_SPECS.get(name) or direct_surface_spec(name, obs_names, row.get("units", ""))
        else:
            spec = PROFILE_OBS_SPECS.get(name) or direct_profile_spec(name, obs_names, row.get("units", ""))
        rows.append(spec.obs if spec_available(spec, obs_names) else "")
    out = catalog.copy()
    out["observation"] = rows
    out["has_observation"] = out["observation"].astype(bool)
    return out


def load_catalog(model_file=None, observation_file=None):
    global MODEL_FILE, OBSERVATION_FILE, CATALOG, SURFACE_CATALOG, PROFILE_CATALOG
    if model_file is not None:
        MODEL_FILE = Path(model_file).expanduser().resolve()
    if observation_file is not None:
        OBSERVATION_FILE = Path(observation_file).expanduser().resolve()
    if not MODEL_FILE.exists():
        raise FileNotFoundError(MODEL_FILE)
    if not OBSERVATION_FILE.exists():
        raise FileNotFoundError(OBSERVATION_FILE)
    CATALOG = attach_observation_mapping(variable_catalog(MODEL_FILE), OBSERVATION_FILE)
    SURFACE_CATALOG = CATALOG[CATALOG["category"] == "surface"].reset_index(drop=True)
    PROFILE_CATALOG = CATALOG[CATALOG["category"] == "profile"].reset_index(drop=True)
    print(f"Loaded model catalog from {MODEL_FILE}")
    print(f"Observation file: {OBSERVATION_FILE}")
    print(f"surface variables: {len(SURFACE_CATALOG)}; with observation: {int(SURFACE_CATALOG['has_observation'].sum())}")
    print(f"profile variables: {len(PROFILE_CATALOG)}; with observation: {int(PROFILE_CATALOG['has_observation'].sum())}")
    return CATALOG


catalog = load_catalog(MODEL_FILE, OBSERVATION_FILE)
catalog


## Plot Helpers


In [ ]:
def filled(var_or_array):
    return np.asarray(np.ma.asarray(var_or_array, dtype=np.float64).filled(np.nan), dtype=np.float64)


def load_model_time_axis(ds):
    time = ds.variables["time"]
    days = np.asarray(time[:], dtype=np.float64)
    dates = np.array(
        num2date(days, time.units, getattr(time, "calendar", "standard"), only_use_cftime_datetimes=False),
        dtype=object,
    )
    return days, dates


def parse_bdate(ds):
    value = int(np.asarray(ds.variables["bdate"][...]).item())
    text = str(value)
    if len(text) == 8:
        return datetime(int(text[:4]), int(text[4:6]), int(text[6:8]))
    if len(text) == 6:
        year = int(text[:2])
        year += 1900 if year >= 70 else 2000
        return datetime(year, int(text[2:4]), int(text[4:6]))
    raise ValueError(f"unsupported bdate value: {value}")


def load_obs_dates(ds):
    if "bdate" in ds.variables and "tsec" in ds.variables:
        base = parse_bdate(ds)
        tsec = np.asarray(ds.variables["tsec"][:], dtype=np.float64)
        return np.array([base + timedelta(seconds=float(x)) for x in tsec], dtype=object)
    if "time" in ds.variables:
        return np.asarray(ds.variables["time"][:], dtype=object)
    raise ValueError("observation file has no recognizable time axis")


def obs_relative_days_from_model_origin(obs_dates, origin):
    return np.asarray([(d - origin).total_seconds() / 86400.0 for d in obs_dates], dtype=np.float64)


def interpolate_obs(obs_days, obs_values, target_days):
    obs_values = np.asarray(obs_values, dtype=np.float64)
    finite = np.isfinite(obs_values)
    if finite.sum() < 2:
        return np.full_like(target_days, np.nan, dtype=np.float64)
    return np.interp(target_days, obs_days[finite], obs_values[finite], left=np.nan, right=np.nan)


def reduce_to_time_series(values, dims):
    values = filled(values)
    axes = tuple(i for i, dim in enumerate(dims) if dim != "time")
    if axes:
        return np.nanmean(values, axis=axes)
    return values


def obs_series(ds, expression):
    parts = [part.strip() for part in expression.split("-")]
    values = reduce_to_time_series(ds.variables[parts[0]][:], tuple(ds.variables[parts[0]].dimensions))
    for part in parts[1:]:
        values = values - reduce_to_time_series(ds.variables[part][:], tuple(ds.variables[part].dimensions))
    return values


def profile_level_dim(dims):
    for dim in ("lev", "ilev"):
        if dim in dims:
            return dim
    raise ValueError(f"no lev/ilev dimension in {dims}")


def reduce_to_time_level(values, dims, level_dim):
    values = filled(values)
    time_axis = dims.index("time")
    level_axis = dims.index(level_dim)
    values = np.moveaxis(values, (time_axis, level_axis), (0, 1))
    if values.ndim > 2:
        values = np.nanmean(values, axis=tuple(range(2, values.ndim)))
    return values


def level_values(ds, level_dim):
    if level_dim in ds.variables and np.issubdtype(np.dtype(ds.variables[level_dim].dtype), np.number):
        values = filled(ds.variables[level_dim][:]).squeeze()
        units = getattr(ds.variables[level_dim], "units", "")
        label = f"{level_dim} ({units})" if units else level_dim
        return values, label
    size = len(ds.dimensions[level_dim])
    return np.arange(size, dtype=np.float64), level_dim


def sort_levels_low_to_high(levels, matrix):
    levels = np.asarray(levels, dtype=np.float64)
    order = np.argsort(levels)
    return levels[order], matrix[:, order]


def is_pressure_axis(level_label):
    lowered = level_label.lower()
    return "hpa" in lowered or "pa" in lowered or "pressure" in lowered


def color_range(values, symmetric=False):
    finite = np.asarray(values)[np.isfinite(values)]
    if finite.size == 0:
        return None, None
    if symmetric:
        hi = np.nanpercentile(np.abs(finite), 98)
        if not np.isfinite(hi) or np.isclose(hi, 0):
            return None, None
        return float(-hi), float(hi)
    lo, hi = np.nanpercentile(finite, [2, 98])
    if np.isclose(lo, hi):
        return None, None
    return float(lo), float(hi)


def variable_title(name, var):
    long_name = getattr(var, "long_name", "")
    units = getattr(var, "units", "")
    text = f"<b>{name}</b>"
    if long_name:
        text += f": {long_name}"
    if units:
        text += f"<br><sup>Units: {units}</sup>"
    return text


def surface_spec_for(variable_name):
    obs_names = observation_variable_names(OBSERVATION_FILE)
    with Dataset(MODEL_FILE) as ds:
        units = getattr(ds.variables[variable_name], "units", "") if variable_name in ds.variables else ""
    spec = SURFACE_OBS_SPECS.get(variable_name) or direct_surface_spec(variable_name, obs_names, units)
    return spec if spec_available(spec, obs_names) else None


def profile_spec_for(variable_name):
    obs_names = observation_variable_names(OBSERVATION_FILE)
    with Dataset(MODEL_FILE) as ds:
        units = getattr(ds.variables[variable_name], "units", "") if variable_name in ds.variables else ""
    spec = PROFILE_OBS_SPECS.get(variable_name) or direct_profile_spec(variable_name, obs_names, units)
    return spec if spec_available(spec, obs_names) else None


def load_surface_comparison(variable_name):
    with Dataset(MODEL_FILE) as model, Dataset(OBSERVATION_FILE) as obs:
        model_days, model_dates = load_model_time_axis(model)
        var = model.variables[variable_name]
        dims = tuple(var.dimensions)
        model_values = reduce_to_time_series(var[:], dims)
        units = getattr(var, "units", "")
        title = variable_title(variable_name, var)

        spec = surface_spec_for(variable_name)
        obs_dates = load_obs_dates(obs)
        obs_native = None
        obs_at_model = None
        if spec is not None:
            origin = model_dates[0] - timedelta(days=float(model_days[0]))
            obs_days = obs_relative_days_from_model_origin(obs_dates, origin)
            obs_native = obs_series(obs, spec.obs) * spec.scale_obs + spec.obs_offset
            obs_at_model = interpolate_obs(obs_days, obs_native, model_days)
    return model_dates, model_values, units, title, spec, obs_dates, obs_native, obs_at_model


def load_profile_matrix(variable_name):
    with Dataset(MODEL_FILE) as ds:
        _, time_dates = load_model_time_axis(ds)
        var = ds.variables[variable_name]
        dims = tuple(var.dimensions)
        level_dim = profile_level_dim(dims)
        matrix = reduce_to_time_level(var[:], dims, level_dim)
        levels, level_label = level_values(ds, level_dim)
        levels, matrix = sort_levels_low_to_high(levels, matrix)
        units = getattr(var, "units", "")
        title = variable_title(variable_name, var)
    return time_dates, levels, matrix, level_label, units, title


def model_pressure(ds):
    required = {"P0", "hyam", "hybm", "PS"}
    if not required.issubset(ds.variables):
        raise ValueError("model file does not contain P0/hyam/hybm/PS needed for pressure interpolation")
    p0 = float(np.asarray(ds.variables["P0"][...]))
    hyam = np.asarray(ds.variables["hyam"][:], dtype=np.float64)
    hybm = np.asarray(ds.variables["hybm"][:], dtype=np.float64)
    ps = np.asarray(ds.variables["PS"][:], dtype=np.float64).squeeze()
    return hyam[None, :] * p0 + hybm[None, :] * ps[:, None]


def interp_model_matrix_to_pressure(values, pressure, target_pressure):
    values = np.asarray(values, dtype=np.float64)
    pressure = np.asarray(pressure, dtype=np.float64)
    idx = np.sum(pressure < target_pressure, axis=1)
    valid = (idx > 0) & (idx < pressure.shape[1])
    out = np.full(values.shape[0], np.nan, dtype=np.float64)
    if not valid.any():
        return out
    rows = np.arange(values.shape[0])[valid]
    upper = idx[valid]
    lower = upper - 1
    p0 = pressure[rows, lower]
    p1 = pressure[rows, upper]
    v0 = values[rows, lower]
    v1 = values[rows, upper]
    ok = np.isfinite(p0) & np.isfinite(p1) & np.isfinite(v0) & np.isfinite(v1) & (p1 != p0)
    interp = np.full(rows.shape[0], np.nan, dtype=np.float64)
    interp[ok] = v0[ok] + (target_pressure - p0[ok]) * (v1[ok] - v0[ok]) / (p1[ok] - p0[ok])
    out[rows] = interp
    return out


def reduce_obs_profile(values, dims):
    values = filled(values)
    time_axis = dims.index("time")
    level_axis = dims.index("lev")
    values = np.moveaxis(values, (time_axis, level_axis), (0, 1))
    if values.ndim > 2:
        values = np.nanmean(values, axis=tuple(range(2, values.ndim)))
    return values


def load_profile_comparison_on_obs_levels(variable_name):
    spec = profile_spec_for(variable_name)
    if spec is None:
        return None
    with Dataset(MODEL_FILE) as model, Dataset(OBSERVATION_FILE) as obs:
        model_days, model_dates = load_model_time_axis(model)
        origin = model_dates[0] - timedelta(days=float(model_days[0]))
        obs_dates = load_obs_dates(obs)
        obs_days = obs_relative_days_from_model_origin(obs_dates, origin)
        obs_levels_pa = np.asarray(obs.variables["lev"][:], dtype=np.float64)
        obs_level_label = "lev (hPa)"
        obs_levels_hpa = obs_levels_pa / 100.0

        model_var = model.variables[variable_name]
        model_dims = tuple(model_var.dimensions)
        model_level_dim = profile_level_dim(model_dims)
        model_values = reduce_to_time_level(model_var[:], model_dims, model_level_dim)
        pressure = model_pressure(model)
        if model_values.shape[1] != pressure.shape[1]:
            raise ValueError(f"{variable_name}: model value levels {model_values.shape[1]} do not match pressure levels {pressure.shape[1]}")

        obs_var = obs.variables[spec.obs]
        obs_values = reduce_obs_profile(obs_var[:], tuple(obs_var.dimensions)) * spec.scale_obs + spec.obs_offset
        model_on_obs = np.column_stack([
            interp_model_matrix_to_pressure(model_values, pressure, float(target_pressure))
            for target_pressure in obs_levels_pa
        ])
        obs_on_model = np.column_stack([
            interpolate_obs(obs_days, obs_values[:, idx], model_days)
            for idx in range(obs_values.shape[1])
        ])
        units = getattr(model_var, "units", spec.units)
        title = variable_title(variable_name, model_var)
    return {
        "spec": spec,
        "model_dates": model_dates,
        "obs_dates": obs_dates,
        "levels_hpa": obs_levels_hpa,
        "level_label": obs_level_label,
        "model_on_obs": model_on_obs,
        "obs_native": obs_values,
        "obs_on_model": obs_on_model,
        "diff": model_on_obs - obs_on_model,
        "units": units,
        "title": title,
    }



def nearest_level_index(levels, target):
    levels = np.asarray(levels, dtype=np.float64)
    return int(np.nanargmin(np.abs(levels - float(target))))


## 1. Surface Time Series With Observation Overlay

All model surface variables are available. If an observation mapping exists, the observation is overlaid and the legend names the mapped observation variable.


In [ ]:
def plot_surface_variable(variable_name):
    model_dates, model_values, units, title, spec, obs_dates, obs_native, obs_at_model = load_surface_comparison(variable_name)
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=model_dates,
            y=model_values,
            mode="lines",
            name="model",
            line=dict(color="#1261A6", width=2.0),
            hovertemplate="%{x}<br>model=%{y:.4g}<extra></extra>",
        )
    )
    subtitle = ""
    if spec is not None:
        fig.add_trace(
            go.Scatter(
                x=obs_dates,
                y=obs_native,
                mode="lines",
                name=f"observation: {spec.obs}",
                line=dict(color="black", width=2.6),
                hovertemplate="%{x}<br>obs=%{y:.4g}<extra></extra>",
            )
        )
        subtitle = f"<br><sup>Observation overlay: {spec.obs}. Converted to model units before comparison.</sup>"
    else:
        subtitle = "<br><sup>No mapped observation variable found; showing model only.</sup>"
    fig.update_layout(
        title=dict(text=title + subtitle, x=0.01, xanchor="left", font=dict(size=22)),
        template="plotly_white",
        height=520,
        width=1120,
        hovermode="x unified",
        margin=dict(l=80, r=40, t=120, b=70),
    )
    fig.update_xaxes(title="Time")
    fig.update_yaxes(title=units or variable_name)
    return fig


## 2. Profile Time Series With Observation Overlay

All model profile variables are available. For mapped variables, the model is interpolated to the selected observation pressure level and the observation is overlaid.


In [ ]:
def profile_level_options(variable_name):
    comparison = load_profile_comparison_on_obs_levels(variable_name)
    if comparison is not None:
        options = [(f"{level:g}", int(idx)) for idx, level in enumerate(comparison["levels_hpa"])]
        return options, comparison["level_label"], True
    _, levels, _, level_label, _, _ = load_profile_matrix(variable_name)
    options = [(f"{level:g}", int(idx)) for idx, level in enumerate(levels)]
    return options, level_label, False


def plot_profile_time_series(variable_name, level_index=0):
    comparison = load_profile_comparison_on_obs_levels(variable_name)
    if comparison is not None:
        levels = comparison["levels_hpa"]
        level_index = int(np.clip(level_index, 0, len(levels) - 1))
        level = levels[level_index]
        fig = go.Figure()
        fig.add_trace(
            go.Scatter(
                x=comparison["model_dates"],
                y=comparison["model_on_obs"][:, level_index],
                mode="lines",
                name="model interpolated to obs level",
                line=dict(color="#1261A6", width=2.0),
                hovertemplate=f"%{{x}}<br>{comparison['level_label']}={level:g}<br>model=%{{y:.4g}}<extra></extra>",
            )
        )
        fig.add_trace(
            go.Scatter(
                x=comparison["obs_dates"],
                y=comparison["obs_native"][:, level_index],
                mode="lines",
                name=f"observation: {comparison['spec'].obs}",
                line=dict(color="black", width=2.6),
                hovertemplate=f"%{{x}}<br>{comparison['level_label']}={level:g}<br>obs=%{{y:.4g}}<extra></extra>",
            )
        )
        subtitle = f"<br><sup>{comparison['level_label']}={level:g}; model interpolated to observation pressure level.</sup>"
        units = comparison["units"]
        title = comparison["title"]
    else:
        time_dates, levels, matrix, level_label, units, title = load_profile_matrix(variable_name)
        level_index = int(np.clip(level_index, 0, len(levels) - 1))
        level = levels[level_index]
        fig = go.Figure(
            go.Scatter(
                x=time_dates,
                y=matrix[:, level_index],
                mode="lines",
                name="model",
                line=dict(color="#1261A6", width=2.0),
                hovertemplate=f"%{{x}}<br>{level_label}={level:g}<br>model=%{{y:.4g}}<extra></extra>",
            )
        )
        subtitle = f"<br><sup>{level_label}={level:g}; no mapped observation variable found.</sup>"
    fig.update_layout(
        title=dict(text=title + subtitle, x=0.01, xanchor="left", font=dict(size=22)),
        template="plotly_white",
        height=540,
        width=1120,
        hovermode="x unified",
        margin=dict(l=80, r=40, t=125, b=70),
    )
    fig.update_xaxes(title="Time")
    fig.update_yaxes(title=units or variable_name)
    return fig


## 3. Model Heatmap

This keeps the model-only heatmap behavior from the reference notebook.


In [ ]:
def plot_profile_heatmap(variable_name):
    time_dates, levels, matrix, level_label, units, title = load_profile_matrix(variable_name)
    cmin, cmax = color_range(matrix)
    fig = go.Figure(
        go.Heatmap(
            x=time_dates,
            y=levels,
            z=matrix.T,
            colorscale="Viridis",
            zmin=cmin,
            zmax=cmax,
            colorbar=dict(title=units or variable_name),
            hovertemplate="%{x}<br>level=%{y}<br>model=%{z:.4g}<extra></extra>",
        )
    )
    fig.update_layout(
        title=dict(
            text=title + "<br><sup>Model-only heatmap. Levels are sorted from low numeric level to high numeric level.</sup>",
            x=0.01,
            xanchor="left",
            font=dict(size=22),
        ),
        template="plotly_white",
        height=620,
        width=1120,
        margin=dict(l=80, r=80, t=120, b=70),
    )
    fig.update_xaxes(title="Time")
    fig.update_yaxes(title=level_label, autorange="reversed" if is_pressure_axis(level_label) else True)
    return fig


## 4. Heatmap Diff: Model Minus Observation

For mapped profile variables, this plots `model - observation` after interpolating model profiles to observation pressure levels and interpolating observation time series to the model time axis.


In [ ]:
def plot_profile_diff_heatmap(variable_name):
    comparison = load_profile_comparison_on_obs_levels(variable_name)
    if comparison is None:
        fig = go.Figure()
        fig.add_annotation(
            text=f"No mapped observation profile variable found for {variable_name}.",
            x=0.5,
            y=0.5,
            xref="paper",
            yref="paper",
            showarrow=False,
            font=dict(size=18),
        )
        fig.update_layout(template="plotly_white", height=420, width=1120)
        return fig

    diff = comparison["diff"]
    cmin, cmax = color_range(diff, symmetric=True)
    fig = go.Figure(
        go.Heatmap(
            x=comparison["model_dates"],
            y=comparison["levels_hpa"],
            z=diff.T,
            colorscale="RdBu_r",
            zmin=cmin,
            zmax=cmax,
            zmid=0,
            colorbar=dict(title=comparison["units"] or variable_name),
            hovertemplate="%{x}<br>level=%{y:g} hPa<br>model-obs=%{z:.4g}<extra></extra>",
        )
    )
    fig.update_layout(
        title=dict(
            text=comparison["title"] + f"<br><sup>Model - observation heatmap; observation: {comparison['spec'].obs}; model interpolated to observation pressure levels.</sup>",
            x=0.01,
            xanchor="left",
            font=dict(size=22),
        ),
        template="plotly_white",
        height=620,
        width=1120,
        margin=dict(l=80, r=80, t=125, b=70),
    )
    fig.update_xaxes(title="Time")
    fig.update_yaxes(title=comparison["level_label"], autorange="reversed")
    return fig


## Interactive Browser


In [ ]:
model_path = widgets.Text(value=str(MODEL_FILE), description="Model file", layout=widgets.Layout(width="980px"))
observation_path = widgets.Text(value=str(OBSERVATION_FILE), description="Obs file", layout=widgets.Layout(width="980px"))
reload_button = widgets.Button(description="Load files", button_style="primary", icon="refresh")
load_output = widgets.Output()

surface_dropdown = widgets.Dropdown(description="Surface", layout=widgets.Layout(width="820px"))
surface_output = widgets.Output()

profile_series_dropdown = widgets.Dropdown(description="Profile", layout=widgets.Layout(width="820px"))
profile_series_level = widgets.SelectionSlider(
    options=[("loading", 0)],
    description="lev",
    continuous_update=False,
    readout=True,
    layout=widgets.Layout(width="620px"),
    style={"description_width": "45px"},
)
profile_series_output = widgets.Output()

profile_heatmap_dropdown = widgets.Dropdown(description="Heatmap", layout=widgets.Layout(width="820px"))
profile_heatmap_output = widgets.Output()

profile_diff_dropdown = widgets.Dropdown(description="Diff", layout=widgets.Layout(width="820px"))
profile_diff_output = widgets.Output()

_updating_profile_levels = False


def dropdown_options(df):
    options = []
    for _, row in df.sort_values("variable").iterrows():
        label = row["variable"]
        if row["has_observation"]:
            label += f" | obs: {row['observation']}"
        elif row["long_name"]:
            label += f" | {row['long_name'][:80]}"
        options.append((label, row["variable"]))
    return options


def preferred_value(options, preferred):
    values = [value for _, value in options]
    for name in preferred:
        if name in values:
            return name
    return values[0] if values else None


def refresh_profile_series_levels(*_):
    global _updating_profile_levels
    if not profile_series_dropdown.value:
        profile_series_level.options = []
        return
    _updating_profile_levels = True
    try:
        options, level_label, has_obs = profile_level_options(profile_series_dropdown.value)
        profile_series_level.description = "hPa" if "hPa" in level_label else level_label.split()[0]
        profile_series_level.options = options
        if options:
            labels_as_float = np.array([float(label) for label, _ in options], dtype=float)
            default_idx = int(np.nanargmin(np.abs(labels_as_float - 500.0))) if np.nanmax(labels_as_float) > 100 else 0
            profile_series_level.value = options[default_idx][1]
    finally:
        _updating_profile_levels = False


def refresh_dropdowns():
    surface_options = dropdown_options(SURFACE_CATALOG)
    profile_options = dropdown_options(PROFILE_CATALOG)
    diff_options = dropdown_options(PROFILE_CATALOG[PROFILE_CATALOG["has_observation"]].reset_index(drop=True))

    surface_dropdown.options = surface_options
    profile_series_dropdown.options = profile_options
    profile_heatmap_dropdown.options = profile_options
    profile_diff_dropdown.options = diff_options

    surface_default = preferred_value(surface_options, ["TREFHT", "TS", "PRECT", "PS"])
    profile_default = preferred_value(profile_options, ["T", "Q", "U", "V", "OMEGA", "RELHUM"])
    diff_default = preferred_value(diff_options, ["T", "Q", "U", "V", "OMEGA", "RELHUM"])
    if surface_default is not None:
        surface_dropdown.value = surface_default
    if profile_default is not None:
        profile_series_dropdown.value = profile_default
        profile_heatmap_dropdown.value = profile_default
    if diff_default is not None:
        profile_diff_dropdown.value = diff_default
    refresh_profile_series_levels()


def reload_files(_=None):
    with load_output:
        clear_output(wait=True)
        try:
            catalog = load_catalog(model_path.value, observation_path.value)
            display(catalog)
            refresh_dropdowns()
        except Exception as exc:
            print(f"Failed to load files: {exc!r}")


def redraw_surface(_=None):
    with surface_output:
        clear_output(wait=True)
        if not surface_dropdown.value:
            print("No surface variables found.")
            return
        display(plot_surface_variable(surface_dropdown.value))


def redraw_profile_series(_=None):
    if _updating_profile_levels:
        return
    with profile_series_output:
        clear_output(wait=True)
        if not profile_series_dropdown.value:
            print("No profile variables found.")
            return
        display(plot_profile_time_series(profile_series_dropdown.value, level_index=profile_series_level.value or 0))


def on_profile_series_variable_change(change=None):
    refresh_profile_series_levels()
    redraw_profile_series()


def redraw_profile_heatmap(_=None):
    with profile_heatmap_output:
        clear_output(wait=True)
        if not profile_heatmap_dropdown.value:
            print("No profile variables found.")
            return
        display(plot_profile_heatmap(profile_heatmap_dropdown.value))


def redraw_profile_diff(_=None):
    with profile_diff_output:
        clear_output(wait=True)
        if not profile_diff_dropdown.value:
            print("No mapped profile variables found.")
            return
        display(plot_profile_diff_heatmap(profile_diff_dropdown.value))


reload_button.on_click(reload_files)
surface_dropdown.observe(redraw_surface, names="value")
profile_series_dropdown.observe(on_profile_series_variable_change, names="value")
profile_series_level.observe(redraw_profile_series, names="value")
profile_heatmap_dropdown.observe(redraw_profile_heatmap, names="value")
profile_diff_dropdown.observe(redraw_profile_diff, names="value")

refresh_dropdowns()
display(widgets.VBox([widgets.HBox([model_path, reload_button]), observation_path, load_output]))
display(widgets.HTML("<h3>1. Surface time series with observation overlay when available</h3>"))
display(widgets.VBox([surface_dropdown, surface_output]))
display(widgets.HTML("<h3>2. Profile time series with observation overlay when available</h3>"))
display(widgets.VBox([profile_series_dropdown, profile_series_level, profile_series_output]))
display(widgets.HTML("<h3>3. Model profile heatmap</h3>"))
display(widgets.VBox([profile_heatmap_dropdown, profile_heatmap_output]))
display(widgets.HTML("<h3>4. Profile heatmap diff: model - observation</h3>"))
display(widgets.VBox([profile_diff_dropdown, profile_diff_output]))

redraw_surface()
redraw_profile_series()
redraw_profile_heatmap()
redraw_profile_diff()


## Save Variable Catalog


In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
safe_name = "".join(ch if ch.isalnum() or ch in "._-" else "_" for ch in MODEL_FILE.stem)
out = OUT_DIR / f"{safe_name}_model_observation_variable_catalog.csv"
CATALOG.to_csv(out, index=False)
print("wrote", out)


## PDF Report

Run this section after loading the model and observation files. By default it builds a full report matching the four notebook sections: all surface time series, all profile selected-level time series, all model profile heatmaps, and profile diff heatmaps where observation is available. Use the `REPORT_*_VARS` lists or the optional time window to make a smaller meeting-ready report.


In [ ]:
from io import BytesIO
import textwrap

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from reportlab.lib import colors
from reportlab.lib.pagesizes import landscape, letter
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import inch
from reportlab.platypus import Image, PageBreak, Paragraph, SimpleDocTemplate, Spacer, Table, TableStyle


REPORT_SURFACE_VARS = None  # None includes all model surface variables.
REPORT_PROFILE_SERIES_VARS = None  # None includes all model profile variables.
REPORT_MODEL_HEATMAP_VARS = None  # None includes all model profile variables.
REPORT_DIFF_HEATMAP_VARS = None  # None includes all mapped profile variables.

# Profile time-series pages use these fixed pressure levels for every profile variable.
REPORT_PROFILE_LEVELS_HPA = [115, 390, 690, 965]

# Optional PDF-only time crop. Use strings like "1997-06-25" or "1997-06-25 12:00".
# Leave as None to keep the full model/observation time ranges.
REPORT_START_TIME = None
REPORT_END_TIME = None

# REPORT_START_TIME = "1997-06-25"
# REPORT_END_TIME = "1997-07-05"


def _available_report_vars(requested, catalog_df):
    available = set(catalog_df["variable"])
    if requested is None:
        return [name for name in catalog_df["variable"]]
    return [name for name in requested if name in available]


def _available_diff_vars(requested=None):
    mapped = PROFILE_CATALOG[PROFILE_CATALOG["has_observation"]]["variable"].tolist()
    if requested is None:
        return mapped
    available = set(mapped)
    return [name for name in requested if name in available]


def _describe_variable(variable_name):
    row = CATALOG[CATALOG["variable"] == variable_name]
    if row.empty:
        return "", ""
    first = row.iloc[0]
    return str(first.get("long_name", "")), str(first.get("units", ""))


def _plain_variable_title(variable_name):
    long_name, units = _describe_variable(variable_name)
    pieces = [variable_name]
    if long_name:
        pieces.append(long_name)
    if units:
        pieces.append(f"units: {units}")
    return " - ".join(pieces)


def _time_window_label(start_time=None, end_time=None):
    start = "start" if start_time is None else str(start_time)
    end = "end" if end_time is None else str(end_time)
    return f"{start} to {end}"


def _time_window_mask(time_dates, start_time=None, end_time=None):
    stamps = pd.to_datetime([str(item) for item in time_dates], format="mixed")
    mask = np.ones(len(stamps), dtype=bool)
    if start_time is not None:
        mask &= stamps >= pd.Timestamp(start_time)
    if end_time is not None:
        mask &= stamps <= pd.Timestamp(end_time)
    if not mask.any():
        raise ValueError(f"time window has no samples: {_time_window_label(start_time, end_time)}")
    return mask


def _apply_time_window(time_dates, values, start_time=None, end_time=None):
    mask = _time_window_mask(time_dates, start_time=start_time, end_time=end_time)
    return np.asarray(time_dates, dtype=object)[mask], np.asarray(values)[mask]


def _wrap_title(ax, title, width=92):
    ax.set_title("\n".join(textwrap.wrap(title, width=width)), loc="left", fontsize=12, pad=10)


def _png_from_figure(fig):
    buffer = BytesIO()
    fig.savefig(buffer, format="png", dpi=180, bbox_inches="tight")
    plt.close(fig)
    buffer.seek(0)
    return buffer


def _format_time_axis(ax):
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
    ax.figure.autofmt_xdate()


def _surface_report_png(variable_name, start_time=None, end_time=None):
    model_dates, model_values, units, _, spec, obs_dates, obs_native, _ = load_surface_comparison(variable_name)
    model_dates, model_values = _apply_time_window(model_dates, model_values, start_time=start_time, end_time=end_time)

    fig, ax = plt.subplots(figsize=(10.8, 5.9))
    ax.plot(model_dates, model_values, color="#1261A6", linewidth=1.35, label="model", zorder=2)
    subtitle = "model only"
    if spec is not None and obs_native is not None:
        obs_dates, obs_native = _apply_time_window(obs_dates, obs_native, start_time=start_time, end_time=end_time)
        ax.plot(obs_dates, obs_native, color="black", linewidth=1.9, label=f"observation: {spec.obs}", zorder=5)
        subtitle = f"model + observation ({spec.obs})"
    _wrap_title(ax, f"Surface: {_plain_variable_title(variable_name)} - {subtitle}")
    ax.set_xlabel("Time")
    ax.set_ylabel(units or variable_name)
    ax.grid(True, color="#d9d9d9", linewidth=0.6, alpha=0.8)
    ax.legend(loc="best", fontsize=8)
    _format_time_axis(ax)
    return _png_from_figure(fig)


def _profile_level_title(position, level_hpa):
    labels = ["top", "middle 1", "middle 2", "bottom"]
    label = labels[position] if position < len(labels) else f"level {position + 1}"
    return f"{label}: {level_hpa:g} hPa"


def _load_model_profile_at_report_pressure_levels(variable_name, target_levels_hpa):
    target_levels_pa = np.asarray(target_levels_hpa, dtype=np.float64) * 100.0
    with Dataset(MODEL_FILE) as model:
        model_days, model_dates = load_model_time_axis(model)
        model_var = model.variables[variable_name]
        model_dims = tuple(model_var.dimensions)
        model_level_dim = profile_level_dim(model_dims)
        model_values = reduce_to_time_level(model_var[:], model_dims, model_level_dim)
        pressure = model_pressure(model)
        if model_values.shape[1] != pressure.shape[1]:
            raise ValueError(f"{variable_name}: model value levels {model_values.shape[1]} do not match pressure levels {pressure.shape[1]}")
        matrix = np.column_stack([
            interp_model_matrix_to_pressure(model_values, pressure, float(target_pressure))
            for target_pressure in target_levels_pa
        ])
        units = getattr(model_var, "units", "")
    return model_dates, matrix, units


def _profile_series_report_png(variable_name, start_time=None, end_time=None):
    target_levels = [float(level) for level in REPORT_PROFILE_LEVELS_HPA]
    comparison = load_profile_comparison_on_obs_levels(variable_name)
    fig, axes = plt.subplots(2, 2, figsize=(10.8, 5.9), sharex=True)
    axes = axes.ravel()

    if comparison is not None:
        levels = comparison["levels_hpa"]
        units = comparison["units"]
        title = f"Profile selected-level time series: {_plain_variable_title(variable_name)} - model + observation"
        for pos, target_level in enumerate(target_levels):
            ax = axes[pos]
            idx = nearest_level_index(levels, target_level)
            actual_level = float(levels[idx])
            model_dates, model_values = _apply_time_window(
                comparison["model_dates"], comparison["model_on_obs"][:, idx], start_time=start_time, end_time=end_time
            )
            obs_dates, obs_values = _apply_time_window(
                comparison["obs_dates"], comparison["obs_native"][:, idx], start_time=start_time, end_time=end_time
            )
            ax.plot(model_dates, model_values, color="#1261A6", linewidth=1.15, label="model", zorder=2)
            ax.plot(obs_dates, obs_values, color="black", linewidth=1.65, label=f"observation: {comparison['spec'].obs}", zorder=5)
            title_level = target_level if np.isclose(actual_level, target_level) else actual_level
            ax.set_title(_profile_level_title(pos, title_level), fontsize=9, loc="left")
    else:
        model_dates, matrix, units = _load_model_profile_at_report_pressure_levels(variable_name, target_levels)
        title = f"Profile selected-level time series: {_plain_variable_title(variable_name)} - model only"
        for pos, target_level in enumerate(target_levels):
            ax = axes[pos]
            shown_dates, series = _apply_time_window(model_dates, matrix[:, pos], start_time=start_time, end_time=end_time)
            ax.plot(shown_dates, series, color="#1261A6", linewidth=1.15, label="model", zorder=2)
            ax.set_title(_profile_level_title(pos, target_level), fontsize=9, loc="left")

    for ax in axes[len(target_levels):]:
        ax.set_visible(False)
    for pos, ax in enumerate(axes[:len(target_levels)]):
        ax.grid(True, color="#d9d9d9", linewidth=0.5, alpha=0.8)
        ax.tick_params(axis="both", labelsize=8)
        if pos in (0, 2):
            ax.set_ylabel(units or variable_name, fontsize=9)
        if pos in (2, 3):
            ax.set_xlabel("Time", fontsize=9)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
    if target_levels:
        axes[0].legend(loc="best", fontsize=7)
    fig.suptitle("\n".join(textwrap.wrap(title, width=98)), fontsize=12, x=0.02, ha="left")
    fig.autofmt_xdate(rotation=30)
    fig.tight_layout(rect=(0, 0, 1, 0.93))
    return _png_from_figure(fig)


def _model_heatmap_report_png(variable_name, start_time=None, end_time=None):
    time_dates, levels, matrix, level_label, units, _ = load_profile_matrix(variable_name)
    time_dates, matrix = _apply_time_window(time_dates, matrix, start_time=start_time, end_time=end_time)
    vmin, vmax = color_range(matrix)

    fig, ax = plt.subplots(figsize=(10.8, 5.9))
    mesh = ax.pcolormesh(time_dates, levels, matrix.T, shading="auto", cmap="viridis", vmin=vmin, vmax=vmax)
    if is_pressure_axis(level_label):
        ax.invert_yaxis()
    _wrap_title(ax, f"Model heatmap: {_plain_variable_title(variable_name)}")
    ax.set_xlabel("Time")
    ax.set_ylabel(level_label)
    _format_time_axis(ax)
    cbar = fig.colorbar(mesh, ax=ax, pad=0.02)
    cbar.set_label(units or variable_name)
    return _png_from_figure(fig)


def _diff_heatmap_report_png(variable_name, start_time=None, end_time=None):
    comparison = load_profile_comparison_on_obs_levels(variable_name)
    if comparison is None:
        return None
    time_dates, diff = _apply_time_window(comparison["model_dates"], comparison["diff"], start_time=start_time, end_time=end_time)
    vmin, vmax = color_range(diff, symmetric=True)

    fig, ax = plt.subplots(figsize=(10.8, 5.9))
    mesh = ax.pcolormesh(time_dates, comparison["levels_hpa"], diff.T, shading="auto", cmap="RdBu_r", vmin=vmin, vmax=vmax)
    ax.invert_yaxis()
    _wrap_title(ax, f"Diff heatmap: {_plain_variable_title(variable_name)} - model minus observation ({comparison['spec'].obs})")
    ax.set_xlabel("Time")
    ax.set_ylabel(comparison["level_label"])
    _format_time_axis(ax)
    cbar = fig.colorbar(mesh, ax=ax, pad=0.02)
    cbar.set_label(comparison["units"] or variable_name)
    return _png_from_figure(fig)


def _add_header(story, title, subtitle=None):
    styles = getSampleStyleSheet()
    story.append(Paragraph(title, styles["Title"]))
    if subtitle:
        story.append(Paragraph(subtitle, styles["BodyText"]))
    story.append(Spacer(1, 0.18 * inch))


def _add_png_page(story, title, png_buffer):
    _add_header(story, title)
    story.append(Image(png_buffer, width=9.8 * inch, height=5.35 * inch))
    story.append(PageBreak())


def _summary_table(story, surface_vars, profile_series_vars, model_heatmap_vars, diff_heatmap_vars, start_time=None, end_time=None):
    styles = getSampleStyleSheet()
    _add_header(
        story,
        "ARM97 Model Output With Observation Report",
        f"Model file: {MODEL_FILE}<br/>Observation file: {OBSERVATION_FILE}<br/>Time window: {_time_window_label(start_time, end_time)}",
    )
    rows = [
        ["Section", "Pages"],
        ["Surface time series", str(len(surface_vars))],
        ["Profile time series selected levels", str(len(profile_series_vars))],
        ["Model profile heatmaps", str(len(model_heatmap_vars))],
        ["Model - observation diff heatmaps", str(len(diff_heatmap_vars))],
        ["Total pages including summary", str(1 + len(surface_vars) + len(profile_series_vars) + len(model_heatmap_vars) + len(diff_heatmap_vars))],
    ]
    table = Table(rows, colWidths=[3.6 * inch, 6.2 * inch])
    table.setStyle(
        TableStyle(
            [
                ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#1f4e79")),
                ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
                ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
                ("GRID", (0, 0), (-1, -1), 0.35, colors.HexColor("#c8c8c8")),
                ("VALIGN", (0, 0), (-1, -1), "TOP"),
                ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#f5f7fa")]),
            ]
        )
    )
    story.append(table)
    story.append(Spacer(1, 0.25 * inch))

    variable_style = styles["BodyText"]
    variable_style.fontSize = 8
    variable_style.leading = 10
    preview_rows = [["Category", "Variables"]]
    preview_rows.append(["Mapped surface", Paragraph(", ".join(SURFACE_CATALOG.loc[SURFACE_CATALOG["has_observation"], "variable"]), variable_style)])
    preview_rows.append(["Mapped profile", Paragraph(", ".join(diff_heatmap_vars), variable_style)])
    preview = Table(preview_rows, colWidths=[1.7 * inch, 8.1 * inch])
    preview.setStyle(
        TableStyle(
            [
                ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#5b6770")),
                ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
                ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
                ("GRID", (0, 0), (-1, -1), 0.35, colors.HexColor("#c8c8c8")),
                ("VALIGN", (0, 0), (-1, -1), "TOP"),
            ]
        )
    )
    story.append(preview)
    story.append(PageBreak())


def build_model_observation_pdf_report(
    output_pdf=None,
    surface_vars=None,
    profile_series_vars=None,
    model_heatmap_vars=None,
    diff_heatmap_vars=None,
    start_time=None,
    end_time=None,
):
    surface_vars = _available_report_vars(surface_vars if surface_vars is not None else REPORT_SURFACE_VARS, SURFACE_CATALOG)
    profile_series_vars = _available_report_vars(
        profile_series_vars if profile_series_vars is not None else REPORT_PROFILE_SERIES_VARS, PROFILE_CATALOG
    )
    model_heatmap_vars = _available_report_vars(
        model_heatmap_vars if model_heatmap_vars is not None else REPORT_MODEL_HEATMAP_VARS, PROFILE_CATALOG
    )
    diff_heatmap_vars = _available_diff_vars(diff_heatmap_vars if diff_heatmap_vars is not None else REPORT_DIFF_HEATMAP_VARS)

    safe_name = "".join(ch if ch.isalnum() or ch in "._-" else "_" for ch in MODEL_FILE.stem)
    if output_pdf is None:
        OUT_DIR.mkdir(parents=True, exist_ok=True)
        output_pdf = OUT_DIR / f"{safe_name}_model_observation_all_variables_report.pdf"
    output_pdf = Path(output_pdf).expanduser().resolve()
    output_pdf.parent.mkdir(parents=True, exist_ok=True)

    story = []
    _summary_table(
        story,
        surface_vars,
        profile_series_vars,
        model_heatmap_vars,
        diff_heatmap_vars,
        start_time=start_time,
        end_time=end_time,
    )

    for variable_name in surface_vars:
        _add_png_page(story, f"Surface: {variable_name}", _surface_report_png(variable_name, start_time=start_time, end_time=end_time))

    for variable_name in profile_series_vars:
        _add_png_page(
            story,
            f"Profile time series: {variable_name}",
            _profile_series_report_png(
                variable_name,
                start_time=start_time,
                end_time=end_time,
            ),
        )

    for variable_name in model_heatmap_vars:
        _add_png_page(story, f"Model heatmap: {variable_name}", _model_heatmap_report_png(variable_name, start_time=start_time, end_time=end_time))

    for variable_name in diff_heatmap_vars:
        png = _diff_heatmap_report_png(variable_name, start_time=start_time, end_time=end_time)
        if png is not None:
            _add_png_page(story, f"Diff heatmap: {variable_name}", png)

    if story and isinstance(story[-1], PageBreak):
        story.pop()

    doc = SimpleDocTemplate(
        str(output_pdf),
        pagesize=landscape(letter),
        rightMargin=0.45 * inch,
        leftMargin=0.45 * inch,
        topMargin=0.38 * inch,
        bottomMargin=0.38 * inch,
    )
    doc.build(story)
    total_pages = 1 + len(surface_vars) + len(profile_series_vars) + len(model_heatmap_vars) + len(diff_heatmap_vars)
    print(f"wrote {output_pdf}")
    print(f"pages: {total_pages}")
    return output_pdf


REPORT_PDF = build_model_observation_pdf_report(start_time=REPORT_START_TIME, end_time=REPORT_END_TIME)
REPORT_PDF
